In [2]:
from pathlib import Path
import pandas as pd

PROJECT = Path(
    "/home/samuel-logan/uni/ENGG2112/lake-erie-forecast"
)

okeechobee = pd.read_csv(
    PROJECT / "data/processed/okeechobee"
    / "okeechobee_prepared_daily.csv",
    parse_dates=["date"],
)

print("Daily records:", len(okeechobee))
print("Recorded labels:", okeechobee["bloom_observed"].notna().sum())
print("\nRecorded class counts:")
print(okeechobee["bloom_observed"].value_counts().sort_index())

display(okeechobee.head())

Daily records: 3287
Recorded labels: 483

Recorded class counts:
bloom_observed
0.0    234
1.0    249
Name: count, dtype: int64


,date,lake,year,day_of_year,rainfall_mm,rainfall_mm_source,air_temperature_c,air_temperature_c_source,wind_speed_m_s,wind_speed_m_s_source,...,discharge_mean_10d_m3_s,rolling_window_days_10d,discharge_estimated_days_10d,season_sin,season_cos,bloom_observed,bloom_observation_count,bloom_observation_status,has_observed_target,observed_target_source
0,2017-01-01,Okeechobee,2017,1,0.0,teammate_supplied,20.6,teammate_supplied,4.67,teammate_supplied,...,0.799000,1,0,0.000000,1.000000,NaN,NaN,No sampling label available,0,unlabelled_or_uncertain
1,2017-01-02,Okeechobee,2017,2,0.4,teammate_supplied,21.9,teammate_supplied,4.86,teammate_supplied,...,0.806000,2,0,0.017202,0.999852,NaN,NaN,No sampling label available,0,unlabelled_or_uncertain
2,2017-01-03,Okeechobee,2017,3,0.0,teammate_supplied,23.0,teammate_supplied,4.17,teammate_supplied,...,0.555333,3,0,0.034399,0.999408,NaN,NaN,No sampling label available,0,unlabelled_or_uncertain
3,2017-01-04,Okeechobee,2017,4,2.3,teammate_supplied,22.8,teammate_supplied,6.25,teammate_supplied,...,0.489500,4,0,0.051585,0.998669,NaN,NaN,No sampling label available,0,unlabelled_or_uncertain
4,2017-01-05,Okeechobee,2017,5,0.0,teammate_supplied,19.8,teammate_supplied,4.03,teammate_supplied,...,0.431400,5,0,0.068757,0.997633,NaN,NaN,No sampling label available,0,unlabelled_or_uncertain


In [3]:
import numpy as np

inspection = okeechobee.copy()

# Divide each year into consecutive ten-day windows.
year_start = inspection["date"].dt.to_period("Y").dt.start_time
window_number = (inspection["date"].dt.dayofyear - 1) // 10

inspection["window_start"] = (
    year_start + pd.to_timedelta(window_number * 10, unit="D")
)

inspection["has_label"] = inspection["bloom_observed"].notna()
inspection["positive_label"] = inspection["bloom_observed"].eq(1)
inspection["negative_label"] = inspection["bloom_observed"].eq(0)

window_coverage = (
    inspection.groupby("window_start")
    .agg(
        calendar_days=("date", "size"),
        labelled_days=("has_label", "sum"),
        bloom_observed_days=("positive_label", "sum"),
        no_bloom_observed_days=("negative_label", "sum"),
    )
    .reset_index()
)

window_coverage["window_end"] = (
    window_coverage["window_start"] + pd.Timedelta(days=9)
)

# Exclude the short leftover window at the end of each year.
full_windows = window_coverage.loc[
    window_coverage["calendar_days"].eq(10)
].copy()

full_windows["observation_summary"] = np.select(
    [
        full_windows["bloom_observed_days"].gt(0),
        full_windows["labelled_days"].gt(0),
    ],
    [
        "At least one recorded bloom",
        "Only no-bloom observations recorded",
    ],
    default="No definite observations",
)

print("Complete ten-day windows:", len(full_windows))
print("\nObservation summaries:")
print(full_windows["observation_summary"].value_counts())

print("\nLabelled days per ten-day window:")
print(full_windows["labelled_days"].value_counts().sort_index())

display(
    full_windows[
        [
            "window_start",
            "window_end",
            "labelled_days",
            "bloom_observed_days",
            "no_bloom_observed_days",
            "observation_summary",
        ]
    ].head(15)
)

Complete ten-day windows: 324

Observation summaries:
observation_summary
No definite observations               148
At least one recorded bloom            116
Only no-bloom observations recorded     60
Name: count, dtype: int64

Labelled days per ten-day window:
labelled_days
0    148
1     34
2     46
3     54
4     23
5     14
6      5
Name: count, dtype: int64


,window_start,window_end,labelled_days,bloom_observed_days,no_bloom_observed_days,observation_summary
0,2017-01-01,2017-01-10,0,0,0,No definite observations
1,2017-01-11,2017-01-20,0,0,0,No definite observations
2,2017-01-21,2017-01-30,0,0,0,No definite observations
3,2017-01-31,2017-02-09,0,0,0,No definite observations
4,2017-02-10,2017-02-19,0,0,0,No definite observations
5,2017-02-20,2017-03-01,0,0,0,No definite observations
6,2017-03-02,2017-03-11,0,0,0,No definite observations
7,2017-03-12,2017-03-21,0,0,0,No definite observations
8,2017-03-22,2017-03-31,0,0,0,No definite observations
9,2017-04-01,2017-04-10,0,0,0,No definite observations


In [4]:
from pathlib import Path
import pandas as pd

PROJECT = Path(
    "/home/samuel-logan/uni/ENGG2112/lake-erie-forecast"
)

cross_lake_data = pd.read_csv(
    PROJECT / "data/processed/cross_lake_v2"
    / "candidate_four_lake_examples.csv",
    parse_dates=[
        "forecast_start",
        "forecast_end",
        "input_start",
        "input_end",
    ],
)

display(
    cross_lake_data.groupby("lake").agg(
        examples=("bloom_condition", "size"),
        bloom_conditions=("bloom_condition", "sum"),
    )
)

print("Total examples:", len(cross_lake_data))

,examples,bloom_conditions
lake,,
Champlain,71,23
Erie,221,84
Okeechobee,176,116
Upper_Klamath,71,31


Total examples: 539


In [5]:
feature_columns = [
    "rainfall_total_previous_10d_mm",
    "flow_mean_previous_10d_m3_s",
    "level_change_previous_10d_window_m",
    "season_sin",
    "season_cos",
]

lake_folds = {}
split_summary = []

for held_out_lake in sorted(cross_lake_data["lake"].unique()):
    test = cross_lake_data.loc[
        cross_lake_data["lake"].eq(held_out_lake)
    ].copy()

    development = cross_lake_data.loc[
        cross_lake_data["lake"].ne(held_out_lake)
    ].copy()

    # Find the latest year separately for each source lake.
    development["year"] = development["forecast_start"].dt.year
    latest_year = development.groupby("lake")["year"].transform("max")

    validation = development.loc[
        development["year"].eq(latest_year)
    ].copy()

    train = development.loc[
        development["year"].lt(latest_year)
    ].copy()

    # Verify separation and input availability.
    assert held_out_lake not in train["lake"].values
    assert held_out_lake not in validation["lake"].values
    assert test["lake"].eq(held_out_lake).all()

    assert len(train) + len(validation) + len(test) == len(cross_lake_data)

    for split in [train, validation, test]:
        assert split[feature_columns].notna().all().all()
        assert split["input_end"].lt(split["forecast_start"]).all()

    lake_folds[held_out_lake] = {
        "train": train,
        "validation": validation,
        "test": test,
    }

    split_summary.append({
        "held_out_lake": held_out_lake,
        "training_examples": len(train),
        "training_positive": int(train["bloom_condition"].sum()),
        "validation_examples": len(validation),
        "validation_positive": int(
            validation["bloom_condition"].sum()
        ),
        "test_examples": len(test),
        "test_positive": int(test["bloom_condition"].sum()),
    })

split_summary = pd.DataFrame(split_summary)
display(split_summary)

,held_out_lake,training_examples,training_positive,validation_examples,validation_positive,test_examples,test_positive
0,Champlain,397,195,71,36,71,23
1,Erie,218,124,100,46,221,84
2,Okeechobee,284,108,79,30,176,116
3,Upper_Klamath,397,191,71,32,71,31


In [11]:
import numpy as np

erie_manifest = pd.read_csv(
    PROJECT / "data/processed/erie_paired_data.csv",
    parse_dates=["period_start"],
)

with np.load(
    PROJECT / "data/processed/erie_images_64x128.npz"
) as saved:
    erie_images = saved["images"]

image_lookup = erie_manifest[
    ["period_start", "array_index"]
].rename(columns={
    "period_start": "forecast_start",
    "array_index": "erie_image_index",
})

image_lookup["lake"] = "Erie"

assert not image_lookup.duplicated(
    ["lake", "forecast_start"]
).any()

image_summary = []

for held_out_lake, splits in lake_folds.items():
    for split_name, table in splits.items():

        # Remove links from any previous run, including merge suffixes.
        old_link_columns = [
            column for column in table.columns
            if column.startswith("erie_image_index")
            or column == "has_image"
        ]

        clean_table = table.drop(columns=old_link_columns)

        linked = clean_table.merge(
            image_lookup,
            on=["lake", "forecast_start"],
            how="left",
            validate="one_to_one",
        )

        linked["erie_image_index"] = (
            linked["erie_image_index"].astype("Int64")
        )
        linked["has_image"] = linked["erie_image_index"].notna()

        assert len(linked) == len(table)

        indices = linked["erie_image_index"].dropna()
        assert indices.between(0, len(erie_images) - 1).all()

        lake_folds[held_out_lake][split_name] = linked

        image_summary.append({
            "held_out_lake": held_out_lake,
            "split": split_name,
            "examples": len(linked),
            "with_images": int(linked["has_image"].sum()),
        })

display(pd.DataFrame(image_summary))
print("Loaded Erie image array:", erie_images.shape)

,held_out_lake,split,examples,with_images
0,Champlain,train,397,211
1,Champlain,validation,71,7
2,Champlain,test,71,0
3,Erie,train,218,0
4,Erie,validation,100,0
5,Erie,test,221,218
6,Okeechobee,train,284,211
7,Okeechobee,validation,79,7
8,Okeechobee,test,176,0
9,Upper_Klamath,train,397,211


Loaded Erie image array: (237, 3, 64, 128)


In [12]:
import copy
import time
import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

def environmental_branch():
    return nn.Sequential(
        nn.Linear(5, 16),
        nn.ReLU(),
        nn.Linear(16, 8),
        nn.ReLU(),
    )


class CrossLakeMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.environment = environmental_branch()
        self.classifier = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 2),
        )

    def forward(self, environment, images, available):
        return self.classifier(self.environment(environment))


class CrossLakeCombined(nn.Module):
    def __init__(self):
        super().__init__()
        self.environment = environmental_branch()

        self.images = nn.Sequential(
            nn.Conv2d(3, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AvgPool2d(4),
            nn.Flatten(),
            nn.Linear(16 * 4 * 8, 8),
            nn.ReLU(),
        )

        # 8 environmental features + 8 image features
        # + 1 image-availability indicator.
        self.classifier = nn.Sequential(
            nn.Linear(17, 16),
            nn.ReLU(),
            nn.Linear(16, 2),
        )

    def forward(self, environment, images, available):
        effective_available = available

        # Randomly hide half the available images during training.
        if self.training:
            keep = (
                torch.rand_like(available) >= 0.5
            ).float()
            effective_available = available * keep

        environment_features = self.environment(environment)
        image_features = self.images(images)

        # Missing/hidden images contribute zero learned image features.
        image_features = image_features * effective_available

        combined = torch.cat(
            [
                environment_features,
                image_features,
                effective_available,
            ],
            dim=1,
        )

        return self.classifier(combined)

In [13]:
SEED = 42
BATCH_SIZE = 16
MAX_EPOCHS = 100
PATIENCE = 15
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.001

CROSS_RESULTS = PROJECT / "results" / "cross_lake_v2"
CROSS_RESULTS.mkdir(parents=True, exist_ok=True)


def prepare_fold(splits):
    scaler = StandardScaler()
    scaler.fit(splits["train"][feature_columns])

    datasets = {}

    for split_name, table in splits.items():
        environment = scaler.transform(
            table[feature_columns]
        ).astype(np.float32)

        images = np.zeros(
            (len(table), 3, 64, 128),
            dtype=np.float32,
        )

        has_image = table["has_image"].to_numpy(dtype=bool)

        if has_image.any():
            indices = table.loc[
                has_image, "erie_image_index"
            ].to_numpy(dtype=np.int64)

            images[has_image] = erie_images[indices]

        available = has_image.astype(np.float32).reshape(-1, 1)

        labels = table["bloom_condition"].to_numpy(
            dtype=np.int64
        )

        assert np.isfinite(environment).all()
        assert np.isfinite(images).all()

        datasets[split_name] = TensorDataset(
            torch.from_numpy(environment),
            torch.from_numpy(images),
            torch.from_numpy(available),
            torch.from_numpy(labels),
        )

    return scaler, datasets


def fit_cross_lake_model(model, datasets, name):
    torch.manual_seed(SEED)

    generator = torch.Generator().manual_seed(SEED)

    loader = DataLoader(
        datasets["train"],
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
    )

    training_labels = datasets["train"].tensors[-1]
    counts = torch.bincount(
        training_labels, minlength=2
    ).float()

    assert (counts > 0).all(), "Training needs both classes."

    weights = counts.sum() / (2 * counts)
    loss_function = nn.CrossEntropyLoss(weight=weights)

    optimiser = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    best_loss = float("inf")
    best_state = None
    best_epoch = None
    waiting = 0
    history = []
    started = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        numerator = 0.0
        denominator = 0.0

        for environment, images, available, labels in loader:
            optimiser.zero_grad()

            scores = model(environment, images, available)
            loss = loss_function(scores, labels)

            loss.backward()
            optimiser.step()

            batch_weight = weights[labels].sum().item()
            numerator += loss.item() * batch_weight
            denominator += batch_weight

        training_loss = numerator / denominator

        model.eval()
        with torch.no_grad():
            environment, images, available, labels = (
                datasets["validation"].tensors
            )
            validation_loss = loss_function(
                model(environment, images, available),
                labels,
            ).item()

        assert np.isfinite(
            [training_loss, validation_loss]
        ).all()

        history.append({
            "epoch": epoch,
            "training_loss": training_loss,
            "validation_loss": validation_loss,
        })

        if validation_loss < best_loss:
            best_loss = validation_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            waiting = 0
        else:
            waiting += 1

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"{name} | epoch {epoch:3d} | "
                f"train {training_loss:.4f} | "
                f"validation {validation_loss:.4f}",
                flush=True,
            )

        if waiting >= PATIENCE:
            break

    model.load_state_dict(best_state)
    model.eval()

    print(
        f"Selected epoch {best_epoch}; "
        f"elapsed {time.time() - started:.1f}s\n",
        flush=True,
    )

    return pd.DataFrame(history), best_epoch, weights


def classification_metrics(actual, predicted):
    tn, fp, fn, tp = confusion_matrix(
        actual, predicted, labels=[0, 1]
    ).ravel()

    return {
        "accuracy": accuracy_score(actual, predicted),
        "precision": precision_score(
            actual, predicted, zero_division=0
        ),
        "recall": recall_score(
            actual, predicted, zero_division=0
        ),
        "F1": f1_score(actual, predicted, zero_division=0),
        "correct_warnings": int(tp),
        "false_alarms": int(fp),
        "missed_blooms": int(fn),
        "correct_low_conditions": int(tn),
    }

In [14]:
cross_lake_runs = {}
validation_rows = []

for held_out_lake, splits in lake_folds.items():
    print(f"\nHELD-OUT LAKE: {held_out_lake}", flush=True)

    scaler, datasets = prepare_fold(splits)

    cross_lake_runs[held_out_lake] = {
        "scaler": scaler,
        "datasets": datasets,
        "models": {},
        "histories": {},
    }

    actual = datasets["validation"].tensors[-1].numpy()
    training_labels = datasets["train"].tensors[-1].numpy()

    majority_class = np.bincount(
        training_labels, minlength=2
    ).argmax()

    baseline_predictions = np.full_like(
        actual, majority_class
    )

    validation_rows.append({
        "held_out_lake": held_out_lake,
        "model": "Training majority baseline",
        "validation_examples": len(actual),
        "selected_epoch": None,
        **classification_metrics(actual, baseline_predictions),
    })

    model_specs = [("Environmental MLP", CrossLakeMLP)]

    # Combined training is possible only when training images exist.
    if splits["train"]["has_image"].any():
        model_specs.append(
            ("MLP + CNN", CrossLakeCombined)
        )

    for model_name, model_class in model_specs:
        torch.manual_seed(SEED)
        model = model_class()

        history, best_epoch, weights = fit_cross_lake_model(
            model,
            datasets,
            name=f"{held_out_lake}: {model_name}",
        )

        with torch.no_grad():
            environment, images, available, labels = (
                datasets["validation"].tensors
            )

            predictions = model(
                environment, images, available
            ).argmax(dim=1).numpy()

        validation_rows.append({
            "held_out_lake": held_out_lake,
            "model": model_name,
            "validation_examples": len(actual),
            "selected_epoch": best_epoch,
            **classification_metrics(actual, predictions),
        })

        run = cross_lake_runs[held_out_lake]
        run["models"][model_name] = model
        run["histories"][model_name] = history

        filename = (
            f"{held_out_lake}_{model_name}"
            .replace(" ", "_")
            .replace("+", "plus")
        )

        history.to_csv(
            CROSS_RESULTS / f"{filename}_history.csv",
            index=False,
        )

        torch.save(
            {
                "model_state": model.state_dict(),
                "model_name": model_name,
                "held_out_lake": held_out_lake,
                "feature_columns": feature_columns,
                "scaler_mean": scaler.mean_.tolist(),
                "scaler_scale": scaler.scale_.tolist(),
                "class_weights": weights.tolist(),
                "best_epoch": best_epoch,
                "image_hiding_probability": (
                    0.5 if model_name == "MLP + CNN" else 0.0
                ),
            },
            CROSS_RESULTS / f"{filename}.pt",
        )

cross_lake_validation_results = pd.DataFrame(validation_rows)

cross_lake_validation_results.to_csv(
    CROSS_RESULTS / "validation_results.csv",
    index=False,
)

display(cross_lake_validation_results.round(3))


HELD-OUT LAKE: Champlain


/tmp/ipykernel_60305/4203787975.py:50: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  torch.from_numpy(labels),


Champlain: Environmental MLP | epoch   1 | train 0.7023 | validation 0.6948
Champlain: Environmental MLP | epoch  10 | train 0.6758 | validation 0.6514
Champlain: Environmental MLP | epoch  20 | train 0.5974 | validation 0.5220
Champlain: Environmental MLP | epoch  30 | train 0.5656 | validation 0.5211
Champlain: Environmental MLP | epoch  40 | train 0.5450 | validation 0.5138
Champlain: Environmental MLP | epoch  50 | train 0.5345 | validation 0.5140
Selected epoch 38; elapsed 2.1s

Champlain: MLP + CNN | epoch   1 | train 0.6942 | validation 0.6864
Champlain: MLP + CNN | epoch  10 | train 0.6423 | validation 0.6030
Champlain: MLP + CNN | epoch  20 | train 0.5804 | validation 0.4911
Champlain: MLP + CNN | epoch  30 | train 0.5461 | validation 0.4952
Champlain: MLP + CNN | epoch  40 | train 0.5168 | validation 0.5247
Selected epoch 25; elapsed 12.2s


HELD-OUT LAKE: Erie
Erie: Environmental MLP | epoch   1 | train 0.7058 | validation 0.7273
Erie: Environmental MLP | epoch  10 | train 0

,held_out_lake,model,validation_examples,selected_epoch,accuracy,precision,recall,F1,correct_warnings,false_alarms,missed_blooms,correct_low_conditions
0,Champlain,Training majority baseline,71,NaN,0.493,0.000,0.000,0.000,0,0,36,35
1,Champlain,Environmental MLP,71,38.0,0.789,0.800,0.778,0.789,28,7,8,28
2,Champlain,MLP + CNN,71,25.0,0.845,0.838,0.861,0.849,31,6,5,29
3,Erie,Training majority baseline,100,NaN,0.460,0.460,1.000,0.630,46,54,0,0
4,Erie,Environmental MLP,100,87.0,0.860,0.864,0.826,0.844,38,6,8,48
5,Okeechobee,Training majority baseline,79,NaN,0.620,0.000,0.000,0.000,0,0,30,49
6,Okeechobee,Environmental MLP,79,65.0,0.848,0.875,0.700,0.778,21,3,9,46
7,Okeechobee,MLP + CNN,79,45.0,0.873,0.917,0.733,0.815,22,2,8,47
8,Upper_Klamath,Training majority baseline,71,NaN,0.549,0.000,0.000,0.000,0,0,32,39
9,Upper_Klamath,Environmental MLP,71,20.0,0.761,0.778,0.656,0.712,21,6,11,33


In [16]:
test_rows = []
prediction_tables = []

for held_out_lake, run in cross_lake_runs.items():
    test_dataset = run["datasets"]["test"]
    environment, images, available, labels = test_dataset.tensors

    actual = labels.numpy()

    # Baseline class is determined using training labels only.
    training_labels = (
        run["datasets"]["train"].tensors[-1].numpy()
    )
    majority_class = np.bincount(
        training_labels, minlength=2
    ).argmax()

    predictions_by_model = {
        "Training majority baseline": np.full_like(
            actual, majority_class
        )
    }
    scores_by_model = {}

    for model_name, model in run["models"].items():
        model.eval()

        with torch.no_grad():
            scores = model(environment, images, available)

            predictions_by_model[model_name] = (
                scores.argmax(dim=1).numpy()
            )

            # Uncalibrated model scores, not verified risk probabilities.
            scores_by_model[model_name] = torch.softmax(
                scores, dim=1
            )[:, 1].numpy()

    test_table = lake_folds[held_out_lake]["test"]

    assert len(test_table) == len(actual)
    assert np.array_equal(
        test_table["bloom_condition"].to_numpy(dtype=np.int64),
        actual,
    )

    for model_name, predictions in predictions_by_model.items():
        test_rows.append({
            "held_out_lake": held_out_lake,
            "model": model_name,
            "test_examples": len(actual),
            **classification_metrics(actual, predictions),
        })

        details = test_table[
            ["lake", "forecast_start", "forecast_end"]
        ].reset_index(drop=True).copy()

        details["model"] = model_name
        details["actual"] = actual
        details["prediction"] = predictions
        details["bloom_score"] = (
            scores_by_model[model_name]
            if model_name in scores_by_model
            else np.nan
        )

        prediction_tables.append(details)

cross_lake_test_results = pd.DataFrame(test_rows)
cross_lake_test_predictions = pd.concat(
    prediction_tables, ignore_index=True
)

display(cross_lake_test_results.round(3))

# Compare models over the SAME three held-out lakes.
# Erie is excluded here because it has no trained combined model.
paired_comparison = (
    cross_lake_test_results.loc[
        cross_lake_test_results["held_out_lake"].ne("Erie")
    ]
    .groupby("model")[["accuracy", "precision", "recall", "F1"]]
    .mean()
)

print("\nEqual-weight averages across the three comparable lakes:")
display(paired_comparison.round(3))

cross_lake_test_results.to_csv(
    CROSS_RESULTS / "held_out_lake_test_results.csv",
    index=False,
)

cross_lake_test_predictions.to_csv(
    CROSS_RESULTS / "held_out_lake_test_predictions.csv",
    index=False,
)

paired_comparison.to_csv(
    CROSS_RESULTS / "three_lake_average_test_results.csv"
)

print("\nTest results and individual predictions saved.")

,held_out_lake,model,test_examples,accuracy,precision,recall,F1,correct_warnings,false_alarms,missed_blooms,correct_low_conditions
0,Champlain,Training majority baseline,71,0.676,0.000,0.000,0.000,0,0,23,48
1,Champlain,Environmental MLP,71,0.761,1.000,0.261,0.414,6,0,17,48
2,Champlain,MLP + CNN,71,0.789,0.654,0.739,0.694,17,9,6,39
3,Erie,Training majority baseline,221,0.380,0.380,1.000,0.551,84,137,0,0
4,Erie,Environmental MLP,221,0.543,0.439,0.726,0.547,61,78,23,59
5,Okeechobee,Training majority baseline,176,0.341,0.000,0.000,0.000,0,0,116,60
6,Okeechobee,Environmental MLP,176,0.511,0.778,0.362,0.494,42,12,74,48
7,Okeechobee,MLP + CNN,176,0.528,0.800,0.379,0.515,44,11,72,49
8,Upper_Klamath,Training majority baseline,71,0.563,0.000,0.000,0.000,0,0,31,40
9,Upper_Klamath,Environmental MLP,71,0.859,1.000,0.677,0.808,21,0,10,40



Equal-weight averages across the three comparable lakes:


,accuracy,precision,recall,F1
model,,,,
Environmental MLP,0.710,0.926,0.433,0.572
MLP + CNN,0.749,0.818,0.652,0.707
Training majority baseline,0.527,0.000,0.000,0.000



Test results and individual predictions saved.


In [17]:
class AvailabilityOnlyControl(CrossLakeCombined):
    def forward(self, environment, images, available):
        effective_available = available

        # Same hiding rule as the full combined model.
        if self.training:
            keep = (
                torch.rand_like(available) >= 0.5
            ).float()
            effective_available = available * keep

        environment_features = self.environment(environment)

        # Keep the classifier input shape, but remove image content.
        image_features = torch.zeros_like(environment_features)

        combined = torch.cat(
            [
                environment_features,
                image_features,
                effective_available,
            ],
            dim=1,
        )

        return self.classifier(combined)


control_validation_rows = []
control_runs = {}

for held_out_lake, run in cross_lake_runs.items():
    if "MLP + CNN" not in run["models"]:
        continue

    # Same starting weights as the original combined-model run.
    torch.manual_seed(SEED)
    control = AvailabilityOnlyControl()

    history, best_epoch, weights = fit_cross_lake_model(
        control,
        run["datasets"],
        name=f"{held_out_lake}: availability-only control",
    )

    control.eval()

    with torch.no_grad():
        environment, images, available, labels = (
            run["datasets"]["validation"].tensors
        )

        predictions = control(
            environment, images, available
        ).argmax(dim=1).numpy()

    control_validation_rows.append({
        "held_out_lake": held_out_lake,
        "model": "Availability-only control",
        "validation_examples": len(labels),
        "selected_epoch": best_epoch,
        **classification_metrics(labels.numpy(), predictions),
    })

    control_runs[held_out_lake] = {
        "model": control,
        "history": history,
        "best_epoch": best_epoch,
    }

    history.to_csv(
        CROSS_RESULTS / f"{held_out_lake}_control_history.csv",
        index=False,
    )

    torch.save(
        {
            "model_state": control.state_dict(),
            "model_name": "Availability-only control",
            "held_out_lake": held_out_lake,
            "feature_columns": feature_columns,
            "scaler_mean": run["scaler"].mean_.tolist(),
            "scaler_scale": run["scaler"].scale_.tolist(),
            "class_weights": weights.tolist(),
            "best_epoch": best_epoch,
        },
        CROSS_RESULTS / f"{held_out_lake}_control.pt",
    )

control_validation_results = pd.DataFrame(
    control_validation_rows
)

control_validation_results.to_csv(
    CROSS_RESULTS / "control_validation_results.csv",
    index=False,
)

comparison = pd.concat(
    [
        cross_lake_validation_results,
        control_validation_results,
    ],
    ignore_index=True,
)

display(
    comparison.loc[
        comparison["held_out_lake"].ne("Erie"),
        [
            "held_out_lake",
            "model",
            "accuracy",
            "precision",
            "recall",
            "F1",
        ],
    ]
    .sort_values(["held_out_lake", "model"])
    .round(3)
)

Champlain: availability-only control | epoch   1 | train 0.6974 | validation 0.6918
Champlain: availability-only control | epoch  10 | train 0.6428 | validation 0.5913
Champlain: availability-only control | epoch  20 | train 0.5989 | validation 0.4976
Champlain: availability-only control | epoch  30 | train 0.5866 | validation 0.4973
Champlain: availability-only control | epoch  40 | train 0.5725 | validation 0.4970
Champlain: availability-only control | epoch  50 | train 0.5724 | validation 0.4942
Selected epoch 38; elapsed 2.6s

Okeechobee: availability-only control | epoch   1 | train 0.6964 | validation 0.6939
Okeechobee: availability-only control | epoch  10 | train 0.6403 | validation 0.6133
Okeechobee: availability-only control | epoch  20 | train 0.5318 | validation 0.4153
Okeechobee: availability-only control | epoch  30 | train 0.5151 | validation 0.3860
Okeechobee: availability-only control | epoch  40 | train 0.4953 | validation 0.3756
Okeechobee: availability-only control 

,held_out_lake,model,accuracy,precision,recall,F1
11,Champlain,Availability-only control,0.831,0.853,0.806,0.829
1,Champlain,Environmental MLP,0.789,0.800,0.778,0.789
2,Champlain,MLP + CNN,0.845,0.838,0.861,0.849
0,Champlain,Training majority baseline,0.493,0.000,0.000,0.000
12,Okeechobee,Availability-only control,0.861,0.880,0.733,0.800
6,Okeechobee,Environmental MLP,0.848,0.875,0.700,0.778
7,Okeechobee,MLP + CNN,0.873,0.917,0.733,0.815
5,Okeechobee,Training majority baseline,0.620,0.000,0.000,0.000
13,Upper_Klamath,Availability-only control,0.789,0.815,0.688,0.746
9,Upper_Klamath,Environmental MLP,0.761,0.778,0.656,0.712


In [18]:
control_test_rows = []

for held_out_lake, control_run in control_runs.items():
    model = control_run["model"]
    model.eval()

    environment, images, available, labels = (
        cross_lake_runs[held_out_lake]["datasets"]["test"].tensors
    )

    with torch.no_grad():
        predictions = model(
            environment, images, available
        ).argmax(dim=1).numpy()

    control_test_rows.append({
        "held_out_lake": held_out_lake,
        "model": "Availability-only control",
        "test_examples": len(labels),
        **classification_metrics(labels.numpy(), predictions),
    })

control_test_results = pd.DataFrame(control_test_rows)

# Remove previous control rows if this cell is rerun.
original_results = cross_lake_test_results.loc[
    cross_lake_test_results["model"].ne(
        "Availability-only control"
    )
]

test_comparison = pd.concat(
    [original_results, control_test_results],
    ignore_index=True,
)

display(
    test_comparison.loc[
        test_comparison["held_out_lake"].ne("Erie"),
        ["held_out_lake", "model", "accuracy", "precision", "recall", "F1"],
    ]
    .sort_values(["held_out_lake", "model"])
    .round(3)
)

control_test_results.to_csv(
    CROSS_RESULTS / "control_test_results.csv",
    index=False,
)

test_comparison.to_csv(
    CROSS_RESULTS / "test_results_with_control.csv",
    index=False,
)

,held_out_lake,model,accuracy,precision,recall,F1
11,Champlain,Availability-only control,0.761,0.714,0.435,0.541
1,Champlain,Environmental MLP,0.761,1.000,0.261,0.414
2,Champlain,MLP + CNN,0.789,0.654,0.739,0.694
0,Champlain,Training majority baseline,0.676,0.000,0.000,0.000
12,Okeechobee,Availability-only control,0.511,0.778,0.362,0.494
6,Okeechobee,Environmental MLP,0.511,0.778,0.362,0.494
7,Okeechobee,MLP + CNN,0.528,0.800,0.379,0.515
5,Okeechobee,Training majority baseline,0.341,0.000,0.000,0.000
13,Upper_Klamath,Availability-only control,0.887,1.000,0.742,0.852
9,Upper_Klamath,Environmental MLP,0.859,1.000,0.677,0.808


In [19]:
# Gather the revised Erie examples with their image links.
# In this existing fold, Erie appears only in the test table.
erie_table = lake_folds["Erie"]["test"].copy()

erie_year = erie_table["forecast_start"].dt.year
latest_erie_year = erie_year.max()

erie_train = erie_table.loc[
    erie_year.lt(latest_erie_year)
].copy()

erie_validation = erie_table.loc[
    erie_year.eq(latest_erie_year)
].copy()

# The same external-lake examples used in our earlier tests.
external_test = pd.concat(
    [
        lake_folds[lake]["test"]
        for lake in ["Champlain", "Okeechobee", "Upper_Klamath"]
    ],
    ignore_index=True,
)

assert not external_test["lake"].eq("Erie").any()
assert not external_test["has_image"].any()

erie_only_scaler, erie_only_datasets = prepare_fold({
    "train": erie_train,
    "validation": erie_validation,
    "test": external_test,
})

print("Erie-only training examples:", len(erie_train))
print("Erie-only validation examples:", len(erie_validation))
print("External test examples:", len(external_test))

erie_only_models = {}
erie_only_rows = []
erie_only_prediction_tables = []

for model_name, model_class in [
    ("Environmental MLP", CrossLakeMLP),
    ("MLP + CNN", CrossLakeCombined),
]:
    torch.manual_seed(SEED)
    model = model_class()

    history, best_epoch, weights = fit_cross_lake_model(
        model,
        erie_only_datasets,
        name=f"Erie-only: {model_name}",
    )

    erie_only_models[model_name] = model
    model.eval()

    with torch.no_grad():
        environment, images, available, labels = (
            erie_only_datasets["test"].tensors
        )

        predictions = model(
            environment, images, available
        ).argmax(dim=1).numpy()

    actual = labels.numpy()

    for lake in ["Champlain", "Okeechobee", "Upper_Klamath"]:
        mask = external_test["lake"].eq(lake).to_numpy()

        erie_only_rows.append({
            "held_out_lake": lake,
            "model": model_name,
            "training_source": "Erie only",
            "test_examples": int(mask.sum()),
            **classification_metrics(
                actual[mask], predictions[mask]
            ),
        })

    details = external_test[
        ["lake", "forecast_start", "forecast_end"]
    ].copy()
    details["model"] = model_name
    details["actual"] = actual
    details["prediction"] = predictions
    erie_only_prediction_tables.append(details)

    filename = (
        f"Erie_only_{model_name}"
        .replace(" ", "_")
        .replace("+", "plus")
    )

    history.to_csv(
        CROSS_RESULTS / f"{filename}_history.csv",
        index=False,
    )

    torch.save(
        {
            "model_state": model.state_dict(),
            "model_name": model_name,
            "training_source": "Erie only",
            "feature_columns": feature_columns,
            "scaler_mean": erie_only_scaler.mean_.tolist(),
            "scaler_scale": erie_only_scaler.scale_.tolist(),
            "class_weights": weights.tolist(),
            "best_epoch": best_epoch,
        },
        CROSS_RESULTS / f"{filename}.pt",
    )

erie_only_results = pd.DataFrame(erie_only_rows)

multi_lake_results = cross_lake_test_results.loc[
    cross_lake_test_results["held_out_lake"].ne("Erie")
    & cross_lake_test_results["model"].isin(
        ["Environmental MLP", "MLP + CNN"]
    )
].copy()

multi_lake_results["training_source"] = "Three lakes"

training_source_comparison = pd.concat(
    [erie_only_results, multi_lake_results],
    ignore_index=True,
)

display(
    training_source_comparison[
        [
            "held_out_lake",
            "model",
            "training_source",
            "accuracy",
            "precision",
            "recall",
            "F1",
        ]
    ]
    .sort_values(["held_out_lake", "model", "training_source"])
    .round(3)
)

training_source_comparison.to_csv(
    CROSS_RESULTS / "erie_only_vs_multi_lake.csv",
    index=False,
)

pd.concat(
    erie_only_prediction_tables, ignore_index=True
).to_csv(
    CROSS_RESULTS / "erie_only_external_predictions.csv",
    index=False,
)

Erie-only training examples: 214
Erie-only validation examples: 7
External test examples: 318
Erie-only: Environmental MLP | epoch   1 | train 0.6995 | validation 0.7381
Erie-only: Environmental MLP | epoch  10 | train 0.6265 | validation 0.6841
Erie-only: Environmental MLP | epoch  20 | train 0.5683 | validation 0.6619
Erie-only: Environmental MLP | epoch  30 | train 0.5454 | validation 0.6380
Erie-only: Environmental MLP | epoch  40 | train 0.5310 | validation 0.6293
Erie-only: Environmental MLP | epoch  50 | train 0.5225 | validation 0.6127
Erie-only: Environmental MLP | epoch  60 | train 0.5172 | validation 0.6183
Selected epoch 54; elapsed 1.5s

Erie-only: MLP + CNN | epoch   1 | train 0.6960 | validation 0.7164
Erie-only: MLP + CNN | epoch  10 | train 0.6311 | validation 0.6577
Erie-only: MLP + CNN | epoch  20 | train 0.5164 | validation 0.6502
Erie-only: MLP + CNN | epoch  30 | train 0.4983 | validation 0.6987
Selected epoch 19; elapsed 5.8s



,held_out_lake,model,training_source,accuracy,precision,recall,F1
0,Champlain,Environmental MLP,Erie only,0.789,0.833,0.435,0.571
6,Champlain,Environmental MLP,Three lakes,0.761,1.000,0.261,0.414
3,Champlain,MLP + CNN,Erie only,0.549,0.385,0.652,0.484
7,Champlain,MLP + CNN,Three lakes,0.789,0.654,0.739,0.694
1,Okeechobee,Environmental MLP,Erie only,0.466,0.750,0.284,0.412
8,Okeechobee,Environmental MLP,Three lakes,0.511,0.778,0.362,0.494
4,Okeechobee,MLP + CNN,Erie only,0.409,0.562,0.466,0.509
9,Okeechobee,MLP + CNN,Three lakes,0.528,0.800,0.379,0.515
2,Upper_Klamath,Environmental MLP,Erie only,0.732,1.000,0.387,0.558
10,Upper_Klamath,Environmental MLP,Three lakes,0.859,1.000,0.677,0.808


In [20]:
FLOW_COLUMN = "flow_mean_previous_10d_m3_s"
LOG_FLOW_COLUMN = "signed_log_flow_mean_previous_10d"

log_feature_columns = [
    LOG_FLOW_COLUMN if column == FLOW_COLUMN else column
    for column in feature_columns
]

LOG_RESULTS = CROSS_RESULTS / "signed_log_flow"
LOG_RESULTS.mkdir(parents=True, exist_ok=True)

log_flow_runs = {}
log_validation_rows = []

for held_out_lake, splits in lake_folds.items():
    print(
        f"\nSIGNED-LOG FLOW — HELD-OUT LAKE: {held_out_lake}",
        flush=True,
    )

    # Separate copies: preserve the original input tables.
    transformed_tables = {}

    for split_name, table in splits.items():
        transformed = table.copy()

        flow = transformed[FLOW_COLUMN].to_numpy(dtype=float)

        transformed[LOG_FLOW_COLUMN] = (
            np.sign(flow) * np.log1p(np.abs(flow))
        )

        transformed_tables[split_name] = transformed

    # Fit scaling using this fold's TRAINING inputs only.
    log_scaler = StandardScaler()
    log_scaler.fit(
        transformed_tables["train"][log_feature_columns]
    )

    datasets = {}

    for split_name, table in transformed_tables.items():
        environment = log_scaler.transform(
            table[log_feature_columns]
        ).astype(np.float32)

        # Reuse the same images, availability flags and labels.
        original_dataset = (
            cross_lake_runs[held_out_lake]["datasets"][split_name]
        )
        _, images, available, labels = original_dataset.tensors

        assert np.array_equal(
            table["bloom_condition"].to_numpy(dtype=np.int64),
            labels.numpy(),
        )
        assert np.isfinite(environment).all()

        datasets[split_name] = TensorDataset(
            torch.from_numpy(environment),
            images,
            available,
            labels,
        )

    log_flow_runs[held_out_lake] = {
        "scaler": log_scaler,
        "datasets": datasets,
        "models": {},
        "histories": {},
    }

    model_specs = [("Environmental MLP", CrossLakeMLP)]

    if splits["train"]["has_image"].any():
        model_specs.append(("MLP + CNN", CrossLakeCombined))

    for model_name, model_class in model_specs:
        # Match the original experiment's initialisation.
        torch.manual_seed(SEED)
        model = model_class()

        history, best_epoch, weights = fit_cross_lake_model(
            model,
            datasets,
            name=f"{held_out_lake}: signed-log {model_name}",
        )

        model.eval()

        with torch.no_grad():
            environment, images, available, labels = (
                datasets["validation"].tensors
            )

            predictions = model(
                environment, images, available
            ).argmax(dim=1).numpy()

        log_validation_rows.append({
            "held_out_lake": held_out_lake,
            "model": model_name,
            "flow_representation": "Signed-log",
            "selected_epoch": best_epoch,
            **classification_metrics(
                labels.numpy(), predictions
            ),
        })

        run = log_flow_runs[held_out_lake]
        run["models"][model_name] = model
        run["histories"][model_name] = history

        filename = (
            f"{held_out_lake}_{model_name}"
            .replace(" ", "_")
            .replace("+", "plus")
        )

        history.to_csv(
            LOG_RESULTS / f"{filename}_history.csv",
            index=False,
        )

        torch.save(
            {
                "model_state": model.state_dict(),
                "model_name": model_name,
                "held_out_lake": held_out_lake,
                "feature_columns": log_feature_columns,
                "flow_transformation": "sign(Q) * log1p(abs(Q))",
                "scaler_mean": log_scaler.mean_.tolist(),
                "scaler_scale": log_scaler.scale_.tolist(),
                "class_weights": weights.tolist(),
                "best_epoch": best_epoch,
                "seed": SEED,
            },
            LOG_RESULTS / f"{filename}.pt",
        )

log_flow_validation_results = pd.DataFrame(log_validation_rows)

original_validation = cross_lake_validation_results.loc[
    cross_lake_validation_results["model"].isin(
        ["Environmental MLP", "MLP + CNN"]
    )
].copy()

original_validation["flow_representation"] = "Original"

flow_validation_comparison = pd.concat(
    [original_validation, log_flow_validation_results],
    ignore_index=True,
)

display(
    flow_validation_comparison[
        [
            "held_out_lake",
            "model",
            "flow_representation",
            "accuracy",
            "precision",
            "recall",
            "F1",
        ]
    ]
    .sort_values(
        ["held_out_lake", "model", "flow_representation"]
    )
    .round(3)
)

flow_validation_comparison.to_csv(
    LOG_RESULTS / "validation_comparison.csv",
    index=False,
)


SIGNED-LOG FLOW — HELD-OUT LAKE: Champlain
Champlain: signed-log Environmental MLP | epoch   1 | train 0.7019 | validation 0.6959
Champlain: signed-log Environmental MLP | epoch  10 | train 0.6353 | validation 0.6225
Champlain: signed-log Environmental MLP | epoch  20 | train 0.5700 | validation 0.5419
Champlain: signed-log Environmental MLP | epoch  30 | train 0.5577 | validation 0.5358
Champlain: signed-log Environmental MLP | epoch  40 | train 0.5502 | validation 0.5332
Champlain: signed-log Environmental MLP | epoch  50 | train 0.5434 | validation 0.5298
Selected epoch 38; elapsed 2.9s

Champlain: signed-log MLP + CNN | epoch   1 | train 0.6937 | validation 0.6870
Champlain: signed-log MLP + CNN | epoch  10 | train 0.6188 | validation 0.6112
Champlain: signed-log MLP + CNN | epoch  20 | train 0.5425 | validation 0.5144
Champlain: signed-log MLP + CNN | epoch  30 | train 0.5179 | validation 0.5318
Champlain: signed-log MLP + CNN | epoch  40 | train 0.5022 | validation 0.5240
Champl

,held_out_lake,model,flow_representation,accuracy,precision,recall,F1
0,Champlain,Environmental MLP,Original,0.789,0.800,0.778,0.789
7,Champlain,Environmental MLP,Signed-log,0.746,0.765,0.722,0.743
1,Champlain,MLP + CNN,Original,0.845,0.838,0.861,0.849
8,Champlain,MLP + CNN,Signed-log,0.803,0.806,0.806,0.806
2,Erie,Environmental MLP,Original,0.860,0.864,0.826,0.844
9,Erie,Environmental MLP,Signed-log,0.870,0.867,0.848,0.857
3,Okeechobee,Environmental MLP,Original,0.848,0.875,0.700,0.778
10,Okeechobee,Environmental MLP,Signed-log,0.873,0.885,0.767,0.821
4,Okeechobee,MLP + CNN,Original,0.873,0.917,0.733,0.815
11,Okeechobee,MLP + CNN,Signed-log,0.899,0.923,0.800,0.857


In [21]:
log_test_rows = []
log_prediction_tables = []

for held_out_lake, run in log_flow_runs.items():
    environment, images, available, labels = (
        run["datasets"]["test"].tensors
    )
    actual = labels.numpy()

    test_table = lake_folds[held_out_lake]["test"]

    assert np.array_equal(
        test_table["bloom_condition"].to_numpy(dtype=np.int64),
        actual,
    )

    for model_name, model in run["models"].items():
        model.eval()

        with torch.no_grad():
            predictions = model(
                environment, images, available
            ).argmax(dim=1).numpy()

        log_test_rows.append({
            "held_out_lake": held_out_lake,
            "model": model_name,
            "flow_representation": "Signed-log",
            "test_examples": len(actual),
            **classification_metrics(actual, predictions),
        })

        details = test_table[
            ["lake", "forecast_start", "forecast_end"]
        ].reset_index(drop=True).copy()

        details["model"] = model_name
        details["actual"] = actual
        details["prediction"] = predictions
        log_prediction_tables.append(details)

log_flow_test_results = pd.DataFrame(log_test_rows)

original_tests = cross_lake_test_results.loc[
    cross_lake_test_results["model"].isin(
        ["Environmental MLP", "MLP + CNN"]
    )
].copy()

original_tests["flow_representation"] = "Original"

flow_test_comparison = pd.concat(
    [original_tests, log_flow_test_results],
    ignore_index=True,
)

display(
    flow_test_comparison[
        [
            "held_out_lake",
            "model",
            "flow_representation",
            "accuracy",
            "precision",
            "recall",
            "F1",
            "false_alarms",
            "missed_blooms",
        ]
    ]
    .sort_values(
        ["held_out_lake", "model", "flow_representation"]
    )
    .round(3)
)

flow_test_comparison.to_csv(
    LOG_RESULTS / "test_comparison.csv",
    index=False,
)

pd.concat(
    log_prediction_tables, ignore_index=True
).to_csv(
    LOG_RESULTS / "test_predictions.csv",
    index=False,
)

,held_out_lake,model,flow_representation,accuracy,precision,recall,F1,false_alarms,missed_blooms
0,Champlain,Environmental MLP,Original,0.761,1.000,0.261,0.414,0,17
7,Champlain,Environmental MLP,Signed-log,0.789,1.000,0.348,0.516,0,15
1,Champlain,MLP + CNN,Original,0.789,0.654,0.739,0.694,9,6
8,Champlain,MLP + CNN,Signed-log,0.803,0.846,0.478,0.611,2,12
2,Erie,Environmental MLP,Original,0.543,0.439,0.726,0.547,78,23
9,Erie,Environmental MLP,Signed-log,0.502,0.421,0.821,0.556,95,15
3,Okeechobee,Environmental MLP,Original,0.511,0.778,0.362,0.494,12,74
10,Okeechobee,Environmental MLP,Signed-log,0.460,0.744,0.276,0.403,11,84
4,Okeechobee,MLP + CNN,Original,0.528,0.800,0.379,0.515,11,72
11,Okeechobee,MLP + CNN,Signed-log,0.489,0.771,0.319,0.451,11,79


In [22]:
season_indices = [
    feature_columns.index("season_sin"),
    feature_columns.index("season_cos"),
]

season_only_runs = {}
season_validation_rows = []
season_test_rows = []

SEASON_RESULTS = CROSS_RESULTS / "season_only"
SEASON_RESULTS.mkdir(parents=True, exist_ok=True)

for held_out_lake, original_run in cross_lake_runs.items():
    season_datasets = {}

    for split_name, dataset in original_run["datasets"].items():
        environment, images, available, labels = dataset.tensors

        # Keep only seasonal information.
        season_environment = torch.zeros_like(environment)
        season_environment[:, season_indices] = (
            environment[:, season_indices]
        )

        season_datasets[split_name] = TensorDataset(
            season_environment,
            images,
            available,
            labels,
        )

    torch.manual_seed(SEED)
    model = CrossLakeMLP()

    history, best_epoch, weights = fit_cross_lake_model(
        model,
        season_datasets,
        name=f"{held_out_lake}: season-only MLP",
    )

    season_only_runs[held_out_lake] = {
        "model": model,
        "datasets": season_datasets,
        "history": history,
    }

    model.eval()

    # Evaluate fixed checkpoints; no test-based tuning.
    for split_name, result_rows in [
        ("validation", season_validation_rows),
        ("test", season_test_rows),
    ]:
        with torch.no_grad():
            environment, images, available, labels = (
                season_datasets[split_name].tensors
            )

            predictions = model(
                environment, images, available
            ).argmax(dim=1).numpy()

        result_rows.append({
            "held_out_lake": held_out_lake,
            "model": "Season-only MLP",
            "examples": len(labels),
            "selected_epoch": best_epoch,
            **classification_metrics(labels.numpy(), predictions),
        })

    history.to_csv(
        SEASON_RESULTS / f"{held_out_lake}_history.csv",
        index=False,
    )

    scaler = original_run["scaler"]

    torch.save(
        {
            "model_state": model.state_dict(),
            "held_out_lake": held_out_lake,
            "feature_columns": feature_columns,
            "active_feature_columns": ["season_sin", "season_cos"],
            "input_encoding": (
                "Apply saved scaler, then zero all non-season columns"
            ),
            "scaler_mean": scaler.mean_.tolist(),
            "scaler_scale": scaler.scale_.tolist(),
            "class_weights": weights.tolist(),
            "best_epoch": best_epoch,
        },
        SEASON_RESULTS / f"{held_out_lake}_model.pt",
    )

season_validation_results = pd.DataFrame(season_validation_rows)
season_test_results = pd.DataFrame(season_test_rows)

season_validation_results.to_csv(
    SEASON_RESULTS / "validation_results.csv", index=False
)
season_test_results.to_csv(
    SEASON_RESULTS / "test_results.csv", index=False
)

# Compare with the original-flow experiments.
season_comparison = pd.concat(
    [
        cross_lake_test_results,
        control_test_results,
        season_test_results,
    ],
    ignore_index=True,
)

# Avoid duplicate rows if an earlier table already included controls.
season_comparison = season_comparison.drop_duplicates(
    ["held_out_lake", "model"], keep="last"
)

display(
    season_comparison[
        [
            "held_out_lake",
            "model",
            "accuracy",
            "precision",
            "recall",
            "F1",
        ]
    ]
    .sort_values(["held_out_lake", "model"])
    .round(3)
)

season_comparison.to_csv(
    SEASON_RESULTS / "test_comparison.csv", index=False
)

Champlain: season-only MLP | epoch   1 | train 0.7026 | validation 0.6968
Champlain: season-only MLP | epoch  10 | train 0.6888 | validation 0.6794
Champlain: season-only MLP | epoch  20 | train 0.6738 | validation 0.6372
Champlain: season-only MLP | epoch  30 | train 0.6622 | validation 0.6051
Champlain: season-only MLP | epoch  40 | train 0.6584 | validation 0.6140
Selected epoch 30; elapsed 2.0s

Erie: season-only MLP | epoch   1 | train 0.7084 | validation 0.7298
Erie: season-only MLP | epoch  10 | train 0.6771 | validation 0.6811
Erie: season-only MLP | epoch  20 | train 0.5177 | validation 0.4644
Erie: season-only MLP | epoch  30 | train 0.5108 | validation 0.4627
Erie: season-only MLP | epoch  40 | train 0.5095 | validation 0.4670
Selected epoch 31; elapsed 1.6s

Okeechobee: season-only MLP | epoch   1 | train 0.7009 | validation 0.7002
Okeechobee: season-only MLP | epoch  10 | train 0.6200 | validation 0.5954
Okeechobee: season-only MLP | epoch  20 | train 0.5521 | validation 0

,held_out_lake,model,accuracy,precision,recall,F1
11,Champlain,Availability-only control,0.761,0.714,0.435,0.541
1,Champlain,Environmental MLP,0.761,1.000,0.261,0.414
2,Champlain,MLP + CNN,0.789,0.654,0.739,0.694
14,Champlain,Season-only MLP,0.901,0.944,0.739,0.829
0,Champlain,Training majority baseline,0.676,0.000,0.000,0.000
4,Erie,Environmental MLP,0.543,0.439,0.726,0.547
15,Erie,Season-only MLP,0.425,0.386,0.869,0.535
3,Erie,Training majority baseline,0.380,0.380,1.000,0.551
12,Okeechobee,Availability-only control,0.511,0.778,0.362,0.494
6,Okeechobee,Environmental MLP,0.511,0.778,0.362,0.494


In [23]:
from pathlib import Path
import pandas as pd
import shutil

PROJECT = Path(
    "/home/samuel-logan/uni/ENGG2112/lake-erie-forecast"
)
EXPORT = PROJECT / "results" / "team_findings"
EXPORT.mkdir(parents=True, exist_ok=True)

# Only use result tables already created by our experiments.
# Separate filenames keep validation and test results distinct.
tables_to_export = {
    "01_main_validation_results": "cross_lake_validation_results",
    "02_main_test_results": "cross_lake_test_results",
    "03_availability_control_validation": "control_validation_results",
    "04_availability_control_test": "control_test_results",
    "05_erie_only_vs_multi_lake_test": "training_source_comparison",
    "06_season_only_validation": "season_validation_results",
    "07_season_only_test": "season_test_results",
    "08_test_comparison_with_controls": "season_comparison",
}

exported = []
missing = []
tables = {}

for filename, variable_name in tables_to_export.items():
    table = globals().get(variable_name)

    if isinstance(table, pd.DataFrame):
        table.to_csv(EXPORT / f"{filename}.csv", index=False)
        tables[filename] = table.copy()
        exported.append(f"{filename}.csv")
    else:
        missing.append(variable_name)

# Summarise dataset sizes, if the prepared dataset is loaded.
data = globals().get("cross_lake_data")

if (
    isinstance(data, pd.DataFrame)
    and {"lake", "bloom_condition"}.issubset(data.columns)
):
    summary = (
        data.groupby("lake")["bloom_condition"]
        .agg(examples="size", bloom_conditions="sum")
        .reset_index()
    )
    summary["low_conditions"] = (
        summary["examples"] - summary["bloom_conditions"]
    )
    summary.to_csv(EXPORT / "00_dataset_summary.csv", index=False)
    tables["00_dataset_summary"] = summary
    exported.append("00_dataset_summary.csv")

# Provide a readable HTML page containing the exported tables.
# Your teammates can open it in their web browser.
sections = [
    """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>ENGG2112 project findings</title>
<style>
body {
    font-family: Arial, sans-serif;
    max-width: 1200px;
    margin: 35px auto;
    padding: 0 20px;
    line-height: 1.5;
}
table {
    border-collapse: collapse;
    margin-bottom: 30px;
    font-size: 14px;
}
th, td {
    border: 1px solid #ddd;
    padding: 8px;
    text-align: left;
}
th { background: #edf2f7; }
.table-wrapper { overflow-x: auto; }
</style>
</head>
<body>
<h1>ENGG2112: Cross-lake bloom forecasting</h1>
<p>
Goal: assess whether models trained on other lakes can predict
bloom conditions during an upcoming ten-day period at an unseen lake.
</p>
<p>
Each experiment held out one entire lake for testing.
Validation used the latest available year from each of the remaining
lakes; earlier years were used for training.
</p>
<p>
The “held_out_lake” column identifies the experiment.
In a validation table, the results concern the other training lakes,
not the held-out lake.
</p>
<p>
Satellite images were available only for Erie.
For the other three held-out lakes, the combined model used images
during training but received no images at prediction time.
</p>
"""
]

for title, table in sorted(tables.items()):
    sections.append(
        f"<h2>{title.replace('_', ' ')}</h2>"
        '<div class="table-wrapper">'
        + table.to_html(
            index=False,
            border=0,
            float_format=lambda value: f"{value:.3f}",
            escape=True,
        )
        + "</div>"
    )

sections.append("</body></html>")

(EXPORT / "OPEN_ME_results.html").write_text(
    "\n".join(sections), encoding="utf-8"
)

readme = """ENGG2112 — TEAM FINDINGS

START HERE
Open OPEN_ME_results.html in a web browser.
The CSV files contain the same results for Excel or report tables.

PROJECT QUESTION
Can models trained on several lakes predict bloom conditions during
an upcoming ten-day period at an unseen lake, and does including
Erie imagery during training improve this transfer?

DATA
539 forecasting examples across four lakes:
- Erie: 221
- Okeechobee: 176
- Champlain: 71
- Upper Klamath: 71

Each example pairs an upcoming ten-day bloom label with environmental
information from the preceding ten days.

The five environmental inputs are:
- Rainfall total
- Mean river discharge
- Water-level change
- Seasonal sine
- Seasonal cosine

EVALUATION
One entire lake was held out for testing in each experiment.
For the remaining lakes, the latest available year was reserved for
validation and earlier years were used for training.

VALIDATION versus TEST
Validation selected training checkpoints.
Test results measured performance on the excluded lake.
A validation row named "Okeechobee" therefore refers to validation
on the OTHER lakes in the Okeechobee-held-out experiment.

MODELS
Environmental MLP: environmental inputs only.
MLP + CNN: environmental inputs plus image features when available.
Training-majority baseline: always predicts the most common training class.
Season-only MLP: uses only the seasonal inputs.
Availability-only control: removes image features while retaining
the image-availability indicator and combined architecture.

IMAGE LIMITATION
Only Erie had satellite images.
The combined model was evaluated on the other three lakes without
images at prediction time. No combined model was trained in the
Erie-held-out fold because that fold had no training imagery.

MAIN FINDINGS FROM THE ORIGINAL RUN
Across Champlain, Okeechobee and Upper Klamath:
- Environmental MLP average F1: 0.572
- Combined MLP + CNN average F1: 0.707
These are equal-weight averages across the three lakes.

Transfer performance varied substantially between lakes.
The season-only model outperformed the combined model on Champlain.
The signed-log flow experiment did not consistently improve transfer.

INTERPRETATION
These results concern experimental bloom-condition labels, not toxin
concentrations or confirmed water safety.

Source labels and observation coverage were imperfect, and some
supplied environmental records were reconstructed. Performance is
therefore agreement with the prepared dataset, not independently
verified operational forecasting accuracy.

Follow-up experiments were conducted after inspecting test results.
They should be described as exploratory. Results are from a single
random seed and do not establish that imagery always improves transfer.

FILES
Only tables available in the current notebook session were exported.
A missing table does not mean that its experiment was never completed.
The signed-log results remain in results/cross_lake_v2/signed_log_flow
if they were saved there by the earlier experiment.
"""

(EXPORT / "README.txt").write_text(readme, encoding="utf-8")

# Record exactly what this run exported.
(EXPORT / "export_manifest.txt").write_text(
    "Exported tables:\n"
    + "\n".join(exported)
    + "\n\nResult variables not loaded:\n"
    + ("\n".join(missing) if missing else "None")
    + "\n",
    encoding="utf-8",
)

# Package only this run's outputs, avoiding stale files from older exports.
zip_staging = PROJECT / "results" / "team_findings_package"
zip_staging.mkdir(parents=True, exist_ok=True)

current_files = exported + [
    "OPEN_ME_results.html",
    "README.txt",
    "export_manifest.txt",
]

# Use an explicit file list when creating the ZIP.
from zipfile import ZipFile, ZIP_DEFLATED

zip_path = PROJECT / "results" / "team_findings.zip"

with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as archive:
    for filename in current_files:
        archive.write(EXPORT / filename, arcname=f"team_findings/{filename}")

print("Results page:", EXPORT / "OPEN_ME_results.html")
print("Send this ZIP to your teammates:", zip_path)
print(f"\nExported {len(exported)} tables.")

if missing:
    print("\nThese result variables were not loaded:")
    for name in missing:
        print(" -", name)
    print("Previously saved results have not been deleted or changed.")

Results page: /home/samuel-logan/uni/ENGG2112/lake-erie-forecast/results/team_findings/OPEN_ME_results.html
Send this ZIP to your teammates: /home/samuel-logan/uni/ENGG2112/lake-erie-forecast/results/team_findings.zip

Exported 9 tables.
